In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [9]:
dataset1_df = pd.read_csv('train-2.csv')
test_df = pd.read_csv('test-2.csv')
train_df=dataset1_df.copy()

print(f"Training data shape : {train_df.shape}")
print(f"Testing data shape : {test_df.shape}")

train_df.head(10)


Training data shape : (159571, 8)
Testing data shape : (153164, 2)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0
5,00025465d4725e87,"""\n\nCongratulations from me as well, use the ...",0,0,0,0,0,0
6,0002bcb3da6cb337,COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK,1,1,1,0,1,0
7,00031b1e95af7921,Your vandalism to the Matt Shirvington article...,0,0,0,0,0,0
8,00037261f536c51d,Sorry if the word 'nonsense' was offensive to ...,0,0,0,0,0,0
9,00040093b2687caa,alignment on this subject and which are contra...,0,0,0,0,0,0


In [10]:
train_df = train_df.drop('id', axis=1)
train_df.head(10)

,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0
5,"""\n\nCongratulations from me as well, use the ...",0,0,0,0,0,0
6,COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK,1,1,1,0,1,0
7,Your vandalism to the Matt Shirvington article...,0,0,0,0,0,0
8,Sorry if the word 'nonsense' was offensive to ...,0,0,0,0,0,0
9,alignment on this subject and which are contra...,0,0,0,0,0,0


In [11]:
label_columns = [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]
train_df['Target'] = train_df[label_columns].max(axis=1)
train_df = train_df[['comment_text', 'Target']]
train_df['comment_text'] = train_df['comment_text'].str.lower()

train_df['comment_text'] = train_df['comment_text'].str.replace(r'\s+', ' ', regex=True).str.strip()
test_df = test_df.drop('id', axis=1)
test_df['comment_text'] = test_df['comment_text'].str.replace(r'\s+', ' ', regex=True).str.strip()

train_df.head(50)

,comment_text,Target
0,explanation why the edits made under my userna...,0
1,d'aww! he matches this background colour i'm s...,0
2,"hey man, i'm really not trying to edit war. it...",0
3,""" more i can't make any real suggestions on im...",0
4,"you, sir, are my hero. any chance you remember...",0
5,""" congratulations from me as well, use the too...",0
6,cocksucker before you piss around on my work,1
7,your vandalism to the matt shirvington article...,0
8,sorry if the word 'nonsense' was offensive to ...,0
9,alignment on this subject and which are contra...,0


In [12]:
train_df['Target'].value_counts()


Target
0    143346
1     16225
Name: count, dtype: int64

In [13]:
X = train_df['comment_text']
y = train_df['Target']
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve



vectorizer = TfidfVectorizer(
    max_features=20000,      
    ngram_range=(1, 2),      
    min_df=2,                
    max_df=0.9,
    stop_words='english',
    sublinear_tf=True        
)

X_train_tfidf = vectorizer.fit_transform(X_train)   
X_test_tfidf = vectorizer.transform(X_test)         


model = LogisticRegression(
    class_weight='balanced',  
    max_iter=1000,
    solver='liblinear',        
    C=1.0
)

model.fit(X_train_tfidf, y_train)


y_pred = model.predict(X_test_tfidf)
y_proba = model.predict_proba(X_test_tfidf)[:, 1]

print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.98      0.95      0.96     28670
           1       0.64      0.86      0.74      3245

    accuracy                           0.94     31915
   macro avg       0.81      0.90      0.85     31915
weighted avg       0.95      0.94      0.94     31915

Confusion Matrix:
 [[27136  1534]
 [  463  2782]]
ROC-AUC: 0.9687476480410688


In [15]:
comments = [
    " you bitch",
    "I love  you",
    "i will kill you",
    "You are an idiot",
    "you are my brother",
    "son of a bitch"
]

new_tfidf = vectorizer.transform(comments)
predictions = model.predict(new_tfidf)
probabilities = model.predict_proba(new_tfidf)[:, 1]

for comment, prediction, probability in zip(comments, predictions, probabilities):
    print("Comment:", comment)
    print("Prediction:", "Toxic" if prediction == 1 else "Non-toxic")
    print("Toxicity probability:", round(probability, 3))
    print("----------------------------")

Comment:  you bitch
Prediction: Toxic
Toxicity probability: 1.0
----------------------------
Comment: I love  you
Prediction: Non-toxic
Toxicity probability: 0.213
----------------------------
Comment: i will kill you
Prediction: Toxic
Toxicity probability: 0.986
----------------------------
Comment: You are an idiot
Prediction: Toxic
Toxicity probability: 1.0
----------------------------
Comment: you are my brother
Prediction: Non-toxic
Toxicity probability: 0.211
----------------------------
Comment: son of a bitch
Prediction: Toxic
Toxicity probability: 0.997
----------------------------


In [16]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)  # TF-IDF/CountVectorizer, whatever you used to turn text into features